# Regression Analysis with Python

In [1]:
# Loading data
import pandas as pd
import janitor
pd.set_option("display.max_columns", None)
data_path = "../../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv"
df = pd.read_csv(data_path)
df.head()

,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size
0,BU79786,Washington,2763.519279,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.811147,Two-Door Car,Medsize
1,QZ44356,Arizona,6979.535903,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.464935,Four-Door Car,Medsize
2,AI49188,Nevada,12887.431650,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.472247,Two-Door Car,Medsize
3,WW63253,California,7645.861827,No,Basic,Bachelor,1/20/11,Unemployed,M,0,Suburban,Married,106,18,65,0,7,Corporate Auto,Corporate L2,Offer1,Call Center,529.881344,SUV,Medsize
4,HB64268,Washington,2813.692575,No,Basic,Bachelor,2/3/11,Employed,M,43836,Rural,Single,73,12,44,0,1,Personal Auto,Personal L1,Offer1,Agent,138.130879,Four-Door Car,Medsize


In [2]:
# rename columns to remove spaces and make them to sneak case
df = df.clean_names()
df.head()

,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size
0,BU79786,Washington,2763.519279,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.811147,Two-Door Car,Medsize
1,QZ44356,Arizona,6979.535903,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.464935,Four-Door Car,Medsize
2,AI49188,Nevada,12887.431650,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.472247,Two-Door Car,Medsize
3,WW63253,California,7645.861827,No,Basic,Bachelor,1/20/11,Unemployed,M,0,Suburban,Married,106,18,65,0,7,Corporate Auto,Corporate L2,Offer1,Call Center,529.881344,SUV,Medsize
4,HB64268,Washington,2813.692575,No,Basic,Bachelor,2/3/11,Employed,M,43836,Rural,Single,73,12,44,0,1,Personal Auto,Personal L1,Offer1,Agent,138.130879,Four-Door Car,Medsize


In [3]:
df['engaged'] = df['response'].apply(lambda x: 0 if x == 'No' else 1)

In [4]:
engagement_rate_df = pd.DataFrame(df.groupby('engaged').count()['response'] / df.shape[0] * 100.0)
engagement_rate_df

,response
engaged,
0,85.679877
1,14.320123


In [5]:
engagement_rate_df.T

engaged,0,1
response,85.679877,14.320123


# Sales Channels

In [6]:
engagement_by_sales_channel_df = pd.pivot_table(df,
                                                values='response',
                                                index='sales_channel',
                                                columns='engaged',
                                                aggfunc=len
                                                ).fillna(0.0)
engagement_by_sales_channel_df.columns = ['Not Engaged', 'Engaged']
engagement_by_sales_channel_df


,Not Engaged,Engaged
sales_channel,,
Agent,2811,666
Branch,2273,294
Call Center,1573,192
Web,1169,156


In [7]:
engagement_by_sales_channel_df.reset_index(inplace=True)
engagement_by_sales_channel_df.columns

Index(['sales_channel', 'Not Engaged', 'Engaged'], dtype='str')

In [8]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly import colors

plot_df = engagement_by_sales_channel_df

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Not Engaged (0)', 'Engaged (1)']
)

for i, col_name in enumerate(['Not Engaged', 'Engaged'], start=1):
    fig.add_trace(
        go.Pie(
            labels=plot_df.index,
            values=plot_df[col_name],
            name=col_name,
            textinfo='percent+label',
            marker=dict(colors=colors.qualitative.Pastel)
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Sales Channel Distribution by Engagement',
    template='ggplot2'
)
fig.show()

# Total Claim Amount by Engagement

In [9]:
fig = go.Figure()

for engaged_value in sorted(df["engaged"].unique()):
    fig.add_trace(
        go.Box(
            y=df.loc[df["engaged"] == engaged_value, "total_claim_amount"],
            name=str(engaged_value),
            boxpoints=False  # similar to showfliers=False
        )
    )

fig.update_layout(
    title="Total Claim Amount Distributions by Engagements",
    xaxis_title="Engaged",
    yaxis_title="Total Claim Amount",
    template="ggplot2"
)

fig.show()

In [10]:
fig = go.Figure()

for engaged_value in sorted(df["engaged"].unique()):
    fig.add_trace(
        go.Box(
            y=df.loc[df["engaged"] == engaged_value, "total_claim_amount"],
            name=str(engaged_value),
            boxpoints='outliers'
        )
    )

fig.update_layout(
    title="Total Claim Amount Distributions by Engagements",
    xaxis_title="Engaged",
    yaxis_title="Total Claim Amount",
    template="ggplot2"
)

fig.show()

# Logistic Regression Analysis

## continuous variables

In [11]:
df['income'].dtype

dtype('int64')

In [12]:
df['customer_lifetime_value'].dtype

dtype('float64')

In [13]:
df.describe()

,customer_lifetime_value,income,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,total_claim_amount,engaged
count,9134.000000,9134.000000,9134.000000,9134.000000,9134.000000,9134.000000,9134.000000,9134.000000,9134.000000
mean,8004.940475,37657.380009,93.219291,15.097000,48.064594,0.384388,2.966170,434.088794,0.143201
std,6870.967608,30379.904734,34.407967,10.073257,27.905991,0.910384,2.390182,290.500092,0.350297
min,1898.007675,0.000000,61.000000,0.000000,0.000000,0.000000,1.000000,0.099007,0.000000
25%,3994.251794,0.000000,68.000000,6.000000,24.000000,0.000000,1.000000,272.258244,0.000000
50%,5780.182197,33889.500000,83.000000,14.000000,48.000000,0.000000,2.000000,383.945434,0.000000
75%,8962.167041,62320.000000,109.000000,23.000000,71.000000,0.000000,4.000000,547.514839,0.000000
max,83325.381190,99981.000000,298.000000,35.000000,99.000000,5.000000,9.000000,2893.239678,1.000000


In [14]:
# Create list of continuous variables
continuous_vars = ['customer_lifetime_value',
                   'income', 'monthly_premium_auto',
                   'months_since_last_claim',
                   'months_since_policy_inception',
                   'number_of_open_complaints',
                   'number_of_policies',
                   'total_claim_amount']


In [15]:
import statsmodels.api as sm

X = sm.add_constant(df[continuous_vars])
y = df['engaged']

logit = sm.Logit(y, X).fit()
logit.summary()

Optimization terminated successfully.
         Current function value: 0.409905
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                engaged   No. Observations:                 9134
Model:                          Logit   Df Residuals:                     9125
Method:                           MLE   Df Model:                            8
Date:                Mon, 30 Mar 2026   Pseudo R-squ.:                0.002015
Time:                        13:42:12   Log-Likelihood:                -3744.1
converged:                       True   LL-Null:                       -3751.6
Covariance Type:            nonrobust   LLR p-value:                   0.05685
=================================================================================================
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const                            -1.7866      0.123    -14.476      0.000      -2.029      -1.545
customer_lifetime_value       -6.327e-06   4.86e-06     -1.301      0.193   -1.59e-05     3.2e-06
income                         2.042e-06   1.09e-06      1.869      0.062   -9.89e-08    4.18e-06
monthly_premium_auto             -0.0001      0.001     -0.097      0.922      -0.003       0.002
months_since_last_claim          -0.0045      0.003     -1.503      0.133      -0.010       0.001
months_since_policy_inception     0.0002      0.001      0.198      0.843      -0.002       0.002
number_of_open_complaints        -0.0326      0.034     -0.964      0.335      -0.099       0.034
number_of_policies               -0.0244      0.013     -1.904      0.057      -0.050       0.001
total_claim_amount                0.0003      0.000      1.895      0.058   -9.52e-06       0.001
=================================================================================================
"""

## Categorical variables

In [16]:
# using .factorize() to convert categorical variables into numeric codes
# Example
gender_values, gender_labels = df['gender'].factorize()

In [17]:
gender_values

array([0, 0, 0, ..., 1, 1, 1], shape=(9134,))

In [18]:
gender_labels

Index(['F', 'M'], dtype='str')

In [19]:
# Create categorical for ordinal variable
# 0: High School or Below
# 1: Bachelor
# 2: College
# 3: Master
# 4: Doctor
categories = pd.Categorical(
    df['education'],
    categories=['High School or Below', 'Bachelor', 'College', 'Master', 'Doctor'],
    ordered=True
)


In [20]:
categories.codes

array([1, 1, 1, ..., 1, 2, 2], shape=(9134,), dtype=int8)

In [21]:
# Create new column in the dataframe for encoded data as gender_factorized and education_factorized
df['gender_factorized'] = gender_values
df['education_factorized'] = categories.codes
df.head()

,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,engaged,gender_factorized,education_factorized
0,BU79786,Washington,2763.519279,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.811147,Two-Door Car,Medsize,0,0,1
1,QZ44356,Arizona,6979.535903,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.464935,Four-Door Car,Medsize,0,0,1
2,AI49188,Nevada,12887.431650,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.472247,Two-Door Car,Medsize,0,0,1
3,WW63253,California,7645.861827,No,Basic,Bachelor,1/20/11,Unemployed,M,0,Suburban,Married,106,18,65,0,7,Corporate Auto,Corporate L2,Offer1,Call Center,529.881344,SUV,Medsize,0,1,1
4,HB64268,Washington,2813.692575,No,Basic,Bachelor,2/3/11,Employed,M,43836,Rural,Single,73,12,44,0,1,Personal Auto,Personal L1,Offer1,Agent,138.130879,Four-Door Car,Medsize,0,1,1


In [22]:
# Create Logistic Regression model

X = sm.add_constant(df[['gender_factorized', 'education_factorized']])
y = df['engaged']
logit = sm.Logit(y, X).fit()
logit.summary()

Optimization terminated successfully.
         Current function value: 0.410140
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                engaged   No. Observations:                 9134
Model:                          Logit   Df Residuals:                     9131
Method:                           MLE   Df Model:                            2
Date:                Mon, 30 Mar 2026   Pseudo R-squ.:                0.001442
Time:                        13:53:05   Log-Likelihood:                -3746.2
converged:                       True   LL-Null:                       -3751.6
Covariance Type:            nonrobust   LLR p-value:                  0.004477
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -1.9194      0.056    -34.255      0.000      -2.029      -1.810
gender_factorized        0.0257      0.060      0.430      0.668      -0.091       0.143
education_factorized     0.0894      0.027      3.278      0.001       0.036       0.143
========================================================================================
"""

# Combining Continuous and Categorical Variables in Logistic Regression

In [23]:
# Create list of variables to include in the model
model_vars = ['customer_lifetime_value', 'income', 'monthly_premium_auto',
              'months_since_last_claim', 'months_since_policy_inception',
              'number_of_open_complaints', 'number_of_policies',
              'total_claim_amount','gender_factorized', 'education_factorized']

X = sm.add_constant(df[model_vars])
y = df['engaged']
logit = sm.Logit(y, X).fit()
logit.summary()

Optimization terminated successfully.
         Current function value: 0.409222
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                engaged   No. Observations:                 9134
Model:                          Logit   Df Residuals:                     9123
Method:                           MLE   Df Model:                           10
Date:                Mon, 30 Mar 2026   Pseudo R-squ.:                0.003678
Time:                        13:56:53   Log-Likelihood:                -3737.8
converged:                       True   LL-Null:                       -3751.6
Covariance Type:            nonrobust   LLR p-value:                  0.002095
=================================================================================================
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const                            -1.9204      0.132    -14.587      0.000      -2.178      -1.662
customer_lifetime_value       -6.052e-06   4.87e-06     -1.242      0.214   -1.56e-05    3.49e-06
income                         2.044e-06   1.09e-06      1.868      0.062      -1e-07    4.19e-06
monthly_premium_auto             -0.0005      0.001     -0.381      0.703      -0.003       0.002
months_since_last_claim          -0.0047      0.003     -1.571      0.116      -0.011       0.001
months_since_policy_inception     0.0002      0.001      0.169      0.866      -0.002       0.002
number_of_open_complaints        -0.0345      0.034     -1.021      0.307      -0.101       0.032
number_of_policies               -0.0239      0.013     -1.864      0.062      -0.049       0.001
total_claim_amount                0.0003      0.000      2.352      0.019    5.82e-05       0.001
gender_factorized                 0.0156      0.060      0.259      0.795      -0.102       0.134
education_factorized              0.0979      0.028      3.536      0.000       0.044       0.152
=================================================================================================
"""